# 01 — EEG preprocessing

Preprocessing of the EEG records of healthy controls (HC) and patients with major
depressive disorder (MDD) in three conditions: eyes-closed rest (EC), eyes-open rest (EO)
and auditory oddball task (TASK). The same pipeline is applied to every record:

1. Read the EDF file, keep the 19 channels of the 10-20 system in a fixed order
2. Common average reference
3. FIR notch filter at 50 and 100 Hz; FIR band-pass filter 1–45 Hz
4. Detection and removal of a narrowband acquisition artifact (4.5–6.5 Hz) by
   generalized eigendecomposition (GED)
5. ICA (FastICA, 15 components) with ICLabel; components labeled as eye, heart, muscle,
   line noise or channel noise with probability > 0.7 are removed
6. Segmentation into non-overlapping 2-s windows (512 samples)
7. Rejection of windows with peak-to-peak amplitude > 150 µV in any channel; the original
   position of each retained window is kept so that later steps can form blocks of truly
   consecutive windows

Records flagged in `00_duplicate_audit.ipynb` are excluded (`EXCLUDED_RECORDS`).
Data are kept in volts; no amplitude normalization is applied.

**Run order:** Step 0 (artifact scan, a few minutes) → Step 1 (full preprocessing, a few
hours) → Step 2 (log and quality report).

**Outputs (in `OUTPUT_DIR`):**

| File | Content |
|---|---|
| `classical_<COND>/<ID>_windows.npy` | Retained windows, shape (n_windows, 512, 19), in volts |
| `classical_<COND>/<ID>_window_indices.npy` | Original index of each retained window (before rejection) |
| `classical_<COND>/<ID>_group.npy` | `'control'` or `'mdd'` |
| `channel_order.json` | Channel order and windowing parameters |
| `theta_artifact_scan.csv` | Artifact frequency and eigenvalues of every record (Step 0) |
| `processing_log.json` | Per-record log: artifact removal, ICA, rejection, continuity |

## Setup

In [ ]:
# Google Colab only: install dependencies and mount Google Drive.
# Skip this cell when running locally with requirements.txt installed.
import sys
if 'google.colab' in sys.modules:
    !pip install -q mne==1.13.2 mne-icalabel
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import mne
import mne_icalabel
import numpy as np
import pandas as pd
import scipy
from mne_icalabel import label_components
from scipy import linalg, signal
from scipy.stats import mannwhitneyu

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

print(f'Python {sys.version.split()[0]}')
for module in (mne, mne_icalabel, np, scipy, pd, matplotlib):
    print(f'{module.__name__:12s} {module.__version__}')

## Configuration

In [ ]:
DATA_DIR = 'path/to/original_edf_files'    # folder with the original EDF files
OUTPUT_DIR = 'path/to/preprocessed'        # output folder (read by 02_feature_extraction)

CONDITIONS = ['EC', 'EO', 'TASK']

# --- Filtering ----------------------------------------------------------------
SFREQ = 256          # Hz (all records)
L_FREQ = 1.0         # Hz, band-pass low edge
H_FREQ = 45.0        # Hz, band-pass high edge
NOTCH_FREQS = [50, 100]

# --- Acquisition artifact (narrowband GED) ------------------------------------
THETA_ARTIFACT_BAND = (4.5, 6.5)   # Hz, search range for the peak frequency f0
THETA_ARTIFACT_HALF_BW = 0.5       # Hz, narrow band = f0 ± 0.5 Hz
THETA_ARTIFACT_MAX_COMP = 2        # an oscillation with a phase lag across channels spans 2 dimensions
THETA_ARTIFACT_LAMBDA_MIN = 0.425  # midpoint of the gap in the lambda_1 distribution (checked in Step 0)

# --- ICA ----------------------------------------------------------------------
ICA_N_COMPONENTS = 15
ICA_PROB_THRESHOLD = 0.7
ICA_RANDOM_STATE = 42
ARTIFACT_LABELS = {'eye blink', 'heart beat', 'muscle artifact', 'line noise', 'channel noise'}

# --- Windows and amplitude rejection ------------------------------------------
WINDOW_SEC = 2
WINDOW_SAMPLES = WINDOW_SEC * SFREQ   # 512 samples, no overlap
REJECT_UV = 150.0                     # peak-to-peak threshold, any channel
BLOCK_SIZE = 5                        # 5 consecutive windows = one 10-s block

# --- Channels -----------------------------------------------------------------
# Canonical channel order, used for all saved data and exported to channel_order.json.
EEG_CHANNELS = [
    'EEG Fp1-LE', 'EEG Fp2-LE',
    'EEG F7-LE', 'EEG F3-LE', 'EEG Fz-LE', 'EEG F4-LE', 'EEG F8-LE',
    'EEG T3-LE', 'EEG C3-LE', 'EEG Cz-LE', 'EEG C4-LE', 'EEG T4-LE',
    'EEG T5-LE', 'EEG P3-LE', 'EEG Pz-LE', 'EEG P4-LE', 'EEG T6-LE',
    'EEG O1-LE', 'EEG O2-LE',
]
CHANNEL_NAMES = [ch.replace('EEG ', '').replace('-LE', '') for ch in EEG_CHANNELS]

# --- Records excluded by the duplicate audit (00_duplicate_audit.ipynb) -------
# (group, subject number, condition)
EXCLUDED_RECORDS = (
    {('HC', 30, c) for c in CONDITIONS}                       # identical to HC 27
    | {('MDD', 34, c) for c in CONDITIONS}                    # identical to MDD 33
    | {('HC', s, 'TASK') for s in (1, 2, 3, 7, 10, 11, 12, 17, 29)}   # copies of the TASK record of HC 4
)

## Functions

### File discovery

In [ ]:
def discover_records(data_dir, condition):
    # List the EDF files of one condition, excluding the records flagged in the audit.
    # File names follow '<H|MDD> S<number> <EC|EO|TASK>.edf', sometimes with extra spaces.
    records = []
    for path in sorted(Path(data_dir).glob('*.edf')):
        base = path.name.upper().replace('.EDF', '').strip()
        if not base.endswith(condition):
            continue
        if base.startswith('MDD'):
            group, label = 'MDD', 'mdd'
        elif base.startswith('H'):
            group, label = 'HC', 'control'
        else:
            print(f'  WARNING: unrecognized file name: {path.name}')
            continue
        m = re.search(r'S\s*(\d+)', base)
        if not m:
            print(f'  WARNING: no subject number in {path.name}')
            continue
        number = int(m.group(1))
        if (group, number, condition) in EXCLUDED_RECORDS:
            continue
        records.append({'filepath': str(path), 'filename': path.name,
                        'group': label, 'subject_id': f'{group}_{number}'})
    return records

### Acquisition artifact removal (narrowband GED)

Spatial filters $\mathbf{w}$ are obtained from $\mathbf{S}\mathbf{w} = \lambda\mathbf{R}\mathbf{w}$,
where $\mathbf{S}$ is the covariance of the signal filtered at $f_0 \pm 0.5$ Hz
(4th-order Butterworth, zero phase) and $\mathbf{R}$ the covariance of the broadband signal
(1–45 Hz), regularized by $10^{-6}\,\mathrm{tr}(\mathbf{R})/C$. The eigenvalue $\lambda$ is
the fraction of a component's power that lies in the narrow band. Up to two components with
$\lambda \geq$ `THETA_ARTIFACT_LAMBDA_MIN` are removed with
$\mathbf{X} - \mathbf{A}_k\mathbf{W}_k^\top\mathbf{X}$,
$\mathbf{A}_k = \mathbf{R}\mathbf{W}_k(\mathbf{W}_k^\top\mathbf{R}\mathbf{W}_k)^{-1}$.
The same rule is applied to every record of both groups.

In [ ]:
def remove_theta_artifact(x, sfreq, band=THETA_ARTIFACT_BAND, half_bw=THETA_ARTIFACT_HALF_BW,
                          lambda_min=None, max_comp=THETA_ARTIFACT_MAX_COMP):
    # x: (channels, samples), average reference, filtered 1-45 Hz.
    # lambda_min=None only measures f0 and the eigenvalues (Step 0); x is returned unchanged.
    # Returns the cleaned signal, f0, the three largest eigenvalues and the number of
    # removed components k.
    x = np.asarray(x, dtype=float)
    f, p = signal.welch(x, fs=sfreq, nperseg=int(4 * sfreq), axis=1)
    in_band = (f >= band[0]) & (f <= band[1])
    f0 = float(f[in_band][np.argmax(p[:, in_band].mean(axis=0))])

    sos = signal.butter(4, [f0 - half_bw, f0 + half_bw], btype='band', fs=sfreq, output='sos')
    S = np.cov(signal.sosfiltfilt(sos, x, axis=1))
    R = np.cov(x)
    R = R + 1e-6 * np.trace(R) / len(R) * np.eye(len(R))   # average reference lowers the rank by 1

    lam, W = linalg.eigh(S, R)
    lam, W = lam[::-1], W[:, ::-1]

    k = 0 if lambda_min is None else int(min(max_comp, np.sum(lam >= lambda_min)))
    if k:
        Wk = W[:, :k]
        A = R @ Wk @ np.linalg.inv(Wk.T @ R @ Wk)           # spatial patterns
        x = x - A @ (Wk.T @ x)
        x = x - x.mean(axis=0, keepdims=True)               # keep the exact average reference
    return x, f0, [float(v) for v in lam[:3]], k


def band_power(x, sfreq, lo, hi):
    # Mean Welch PSD within [lo, hi), used for the before/after ratios in the log.
    f, p = signal.welch(x, fs=sfreq, nperseg=int(4 * sfreq), axis=1)
    return float(p[:, (f >= lo) & (f < hi)].mean())

### Preprocessing of one record

In [ ]:
def load_and_filter(filepath, log):
    # Steps 1-3: read, select and order channels, montage, average reference, filters.
    # Shared by Step 0 and Step 1, so that both see exactly the same signal.
    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
    log['duration_seconds'] = float(raw.times[-1])
    log['n_channels_original'] = len(raw.ch_names)

    missing = [ch for ch in EEG_CHANNELS if ch not in raw.ch_names]
    if missing:
        print(f"  WARNING: missing channels in {log['filename']}: {missing}")
        return None

    raw.pick(EEG_CHANNELS)
    raw.reorder_channels(EEG_CHANNELS)
    raw.rename_channels(dict(zip(EEG_CHANNELS, CHANNEL_NAMES)))
    raw.set_channel_types({ch: 'eeg' for ch in raw.ch_names})

    # Montage (needed by ICLabel). Newer MNE versions name T3/T4/T5/T6 as T7/T8/P7/P8.
    old_to_new = {'T3': 'T7', 'T4': 'T8', 'T5': 'P7', 'T6': 'P8'}
    try:
        raw.set_montage('standard_1020', on_missing='raise')
    except Exception:
        raw.rename_channels(old_to_new)
        raw.set_montage('standard_1020', on_missing='warn')
        raw.rename_channels({new: old for old, new in old_to_new.items()})
    raw.reorder_channels(CHANNEL_NAMES)
    assert raw.ch_names == CHANNEL_NAMES, f'unexpected channel order: {raw.ch_names}'

    # ICLabel was trained on average-referenced data; the EDF files use linked ears.
    raw.set_eeg_reference('average', projection=False, verbose=False)

    raw.notch_filter(freqs=NOTCH_FREQS, method='fir', verbose=False)
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, method='fir', fir_design='firwin', verbose=False)
    return raw


def reject_windows(windows):
    # Keep windows with peak-to-peak amplitude below REJECT_UV in all channels.
    ptp = np.ptp(windows, axis=1)                    # (n_windows, n_channels), volts
    return np.all(ptp < REJECT_UV * 1e-6, axis=1)


def count_contiguous_blocks(indices, block_size=BLOCK_SIZE):
    # Number of non-overlapping blocks of `block_size` consecutive windows, from the
    # original indices. Example: [0..4, 8..12] -> 2 blocks; [0, 1, 4, 5, 6] -> 0 blocks.
    indices = np.asarray(indices, dtype=np.int64)
    n_blocks, i = 0, 0
    while i <= len(indices) - block_size:
        if np.all(np.diff(indices[i:i + block_size]) == 1):
            n_blocks += 1
            i += block_size
        else:
            i += 1
    return n_blocks


def preprocess_record(filepath):
    # Full pipeline for one EDF file. Returns (windows, window_indices) or None, and the log.
    log = {
        'filename': os.path.basename(filepath), 'success': False,
        'duration_seconds': 0.0, 'n_channels_original': 0,
        'theta_artifact': None,
        'n_ica_components_total': 0, 'n_ica_components_removed': 0, 'ica_components': [],
        'n_windows_before_rejection': 0, 'n_windows_after_rejection': 0,
        'rejection_rate': 1.0, 'reject_threshold_uv': REJECT_UV,
        'n_temporal_gaps': 0, 'n_contiguous_pairs': 0, 'n_contiguous_blocks_10s': 0,
    }
    try:
        raw = load_and_filter(filepath, log)
        if raw is None:
            return None, log

        # Acquisition artifact
        sfreq = raw.info['sfreq']
        x0 = raw.get_data()
        x1, f0, lam, k = remove_theta_artifact(x0, sfreq, lambda_min=THETA_ARTIFACT_LAMBDA_MIN)
        if k:
            raw._data[:] = x1
        log['theta_artifact'] = {
            'f0': f0, 'lambda': lam, 'k': k, 'lambda_min': THETA_ARTIFACT_LAMBDA_MIN,
            'alpha_ratio': band_power(x1, sfreq, 8, 13) / band_power(x0, sfreq, 8, 13),
            'theta_ratio': band_power(x1, sfreq, 4, 8) / band_power(x0, sfreq, 4, 8),
        }

        # ICA + ICLabel
        ica = mne.preprocessing.ICA(n_components=ICA_N_COMPONENTS, method='fastica',
                                    random_state=ICA_RANDOM_STATE, max_iter='auto')
        ica.fit(raw, verbose=False)
        labels = label_components(raw, ica, method='iclabel')
        exclude = []
        for idx, (label, prob) in enumerate(zip(labels['labels'], labels['y_pred_proba'])):
            drop = label in ARTIFACT_LABELS and prob > ICA_PROB_THRESHOLD
            log['ica_components'].append({'component': idx, 'label': label,
                                          'probability': float(prob), 'excluded': bool(drop)})
            if drop:
                exclude.append(idx)
        ica.exclude = exclude
        ica.apply(raw, verbose=False)
        log['n_ica_components_total'] = int(ica.n_components_)
        log['n_ica_components_removed'] = len(exclude)

        # Non-overlapping 2-s windows: (n_windows, samples, channels)
        data = raw.get_data()
        n_win = data.shape[1] // WINDOW_SAMPLES
        log['n_windows_before_rejection'] = n_win
        if n_win == 0:
            return None, log
        windows = np.stack([data[:, i * WINDOW_SAMPLES:(i + 1) * WINDOW_SAMPLES].T
                            for i in range(n_win)])
        window_indices = np.arange(n_win, dtype=np.int32)   # original position, never renumbered

        # Amplitude rejection
        keep = reject_windows(windows)
        windows, window_indices = windows[keep], window_indices[keep]

        diffs = np.diff(window_indices)
        log['n_contiguous_pairs'] = int(np.sum(diffs == 1))
        log['n_temporal_gaps'] = int(np.sum(diffs > 1))
        log['n_contiguous_blocks_10s'] = count_contiguous_blocks(window_indices)
        log['n_windows_after_rejection'] = int(len(windows))
        log['rejection_rate'] = float(1 - len(windows) / n_win)
        if len(windows) == 0:
            return None, log
        log['success'] = True
        return (windows, window_indices), log

    except Exception as e:
        print(f"  ERROR in {log['filename']}: {e}")
        log['error'] = str(e)
        return None, log

### Processing and saving one condition

In [ ]:
def process_condition(condition):
    records = discover_records(DATA_DIR, condition)
    n_hc = sum(r['group'] == 'control' for r in records)
    print(f"\n{'=' * 70}\n{condition}: {len(records)} files (HC {n_hc}, MDD {len(records) - n_hc})\n{'-' * 70}")

    data, logs = {}, []
    for i, rec in enumerate(records, 1):
        out, log = preprocess_record(rec['filepath'])
        log.update(group=rec['group'], subject_id=rec['subject_id'], condition=condition)
        logs.append(log)
        print(f"[{i:2d}/{len(records)}] {rec['filename']:22s} ", end='')
        if out is None:
            print('FAILED')
            continue
        data[rec['subject_id']] = {'windows': out[0], 'window_indices': out[1], 'group': rec['group']}
        ta = log['theta_artifact']
        removed = [f"IC{c['component']}:{c['label']}" for c in log['ica_components'] if c['excluded']]
        print(f"{log['n_windows_after_rejection']:4d} win | artifact k={ta['k']} "
              f"f0={ta['f0']:.2f} l1={ta['lambda'][0]:.2f} | ICA {len(removed)} | "
              f"rej {log['rejection_rate']:5.1%} | {log['n_temporal_gaps']} gaps | "
              f"{log['n_contiguous_blocks_10s']} blocks | [{', '.join(removed) or 'none'}]")
    return data, logs


def save_condition(data, condition):
    # One folder per condition; windows[i] comes from the original window window_indices[i].
    out_dir = os.path.join(OUTPUT_DIR, f'classical_{condition}')
    os.makedirs(out_dir, exist_ok=True)
    for sid, d in data.items():
        np.save(os.path.join(out_dir, f'{sid}_windows.npy'), d['windows'])
        np.save(os.path.join(out_dir, f'{sid}_window_indices.npy'), d['window_indices'].astype(np.int32))
        np.save(os.path.join(out_dir, f'{sid}_group.npy'), np.array([d['group']], dtype='<U16'))

    # Verify that every subject has its three files and that windows and indices match.
    for sid, d in data.items():
        idx = np.load(os.path.join(out_dir, f'{sid}_window_indices.npy'))
        grp = str(np.load(os.path.join(out_dir, f'{sid}_group.npy'))[0])
        assert len(idx) == len(d['windows']) and np.all(np.diff(idx) > 0), sid
        assert grp == d['group'], sid
        assert os.path.exists(os.path.join(out_dir, f'{sid}_windows.npy')), sid
    print(f'Saved and verified: {len(data)} subjects in {out_dir}')


def save_channel_order():
    with open(os.path.join(OUTPUT_DIR, 'channel_order.json'), 'w') as fh:
        json.dump({'channel_names': CHANNEL_NAMES, 'sfreq': SFREQ,
                   'window_samples': WINDOW_SAMPLES, 'window_seconds': WINDOW_SEC,
                   'overlap': 0.0, 'window_step_samples': WINDOW_SAMPLES}, fh, indent=2)

### Quality report

In [ ]:
def quality_report(logs):
    ok = [lg for lg in logs if lg['success']]
    groups = [('control', 'HC'), ('mdd', 'MDD')]

    print('\nWINDOW REJECTION AND ICA, by condition and group')
    print(f"  {'cond':5s} {'group':5s} {'n':>3s} {'mean rej':>9s} {'median rej':>11s} "
          f"{'max rej':>8s} {'mean ICA removed':>17s}")
    for cond in CONDITIONS:
        rates = {}
        for g, name in groups:
            sub = [lg for lg in ok if lg['condition'] == cond and lg['group'] == g]
            if not sub:
                continue
            r = np.array([lg['rejection_rate'] for lg in sub])
            ica = np.array([lg['n_ica_components_removed'] for lg in sub])
            rates[g] = r
            print(f'  {cond:5s} {name:5s} {len(r):3d} {r.mean():9.1%} {np.median(r):11.1%} '
                  f'{r.max():8.1%} {ica.mean():17.2f}')
        if len(rates) == 2:
            p = mannwhitneyu(rates['control'], rates['mdd'], alternative='two-sided').pvalue
            print(f'  {cond:5s} two-sided Mann-Whitney between groups: p = {p:.4f}')

    print(f'\nACQUISITION ARTIFACT (lambda >= {THETA_ARTIFACT_LAMBDA_MIN})')
    print(f"  {'cond':5s} {'group':5s} {'n':>3s} {'corrected':>10s} {'alpha after/before (median, min)':>34s} "
          f"{'theta after/before (median)':>28s}")
    corrected = []
    for cond in CONDITIONS:
        for g, name in groups:
            sub = [lg for lg in ok if lg['condition'] == cond and lg['group'] == g]
            corr = [lg for lg in sub if lg['theta_artifact']['k'] > 0]
            corrected += corr
            if not sub:
                continue
            if corr:
                ar = np.array([lg['theta_artifact']['alpha_ratio'] for lg in corr])
                tr = np.array([lg['theta_artifact']['theta_ratio'] for lg in corr])
                ratios = f'{np.median(ar):26.3f}, {ar.min():.3f} {np.median(tr):28.3f}'
            else:
                ratios = f"{'-':>34s} {'-':>28s}"
            print(f'  {cond:5s} {name:5s} {len(sub):3d} {len(corr):10d} {ratios}')
    if corrected:
        f0 = np.array([lg['theta_artifact']['f0'] for lg in corrected])
        l2 = np.array([lg['theta_artifact']['lambda'][1] for lg in corrected])
        ar = np.array([lg['theta_artifact']['alpha_ratio'] for lg in corrected])
        print(f'  Total corrected: {len(corrected)} records | f0 {f0.min():.2f}-{f0.max():.2f} Hz '
              f'(median {np.median(f0):.2f}) | lambda_2 {l2.min():.2f}-{l2.max():.2f} | '
              f'alpha ratio {ar.min():.3f}-{ar.max():.3f}')
        print('  Corrected records:')
        for lg in sorted(corrected, key=lambda l: (l['condition'], l['subject_id'])):
            ta = lg['theta_artifact']
            print(f"    {lg['condition']:5s} {lg['subject_id']:8s} f0 {ta['f0']:5.2f} Hz  "
                  f"lambda {ta['lambda'][0]:.2f}/{ta['lambda'][1]:.2f}  k={ta['k']}  "
                  f"alpha x{ta['alpha_ratio']:.3f}  theta x{ta['theta_ratio']:.3f}")

    # Subjects present in all three conditions (the analytic sample is drawn from this core).
    by_cond = {c: {lg['subject_id'] for lg in ok if lg['condition'] == c} for c in CONDITIONS}
    union = set.union(*by_cond.values())
    core = set.intersection(*by_cond.values())
    print('\nCOVERAGE ACROSS CONDITIONS')
    for c, s in by_cond.items():
        print(f'  {c:5s}: {len(s):2d} subjects | missing: {sorted(union - s) or "-"}')
    n_core_hc = sum(s.startswith('HC') for s in core)
    print(f'  Union {len(union)} | present in all conditions {len(core)} '
          f'(HC {n_core_hc}, MDD {len(core) - n_core_hc})')

## Step 0 — Artifact scan

Measures, in every record, the peak frequency $f_0$ and the eigenvalues
$\lambda_1$–$\lambda_3$ of the narrowband GED, without changing the data. The threshold is
set from the distribution of $\lambda_1$ over all records, without group labels: the
distribution is bimodal, and the threshold is the midpoint of the gap between the two modes.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
rows = []
for condition in CONDITIONS:
    for rec in discover_records(DATA_DIR, condition):
        raw = load_and_filter(rec['filepath'], {'filename': rec['filename']})
        if raw is None:
            continue
        _, f0, lam, _ = remove_theta_artifact(raw.get_data(), raw.info['sfreq'])
        rows.append({'condition': condition, 'subject_id': rec['subject_id'],
                     'group': rec['group'], 'file': rec['filename'],
                     'f0': f0, 'lambda1': lam[0], 'lambda2': lam[1], 'lambda3': lam[2]})
scan = pd.DataFrame(rows)
scan.to_csv(os.path.join(OUTPUT_DIR, 'theta_artifact_scan.csv'), index=False)
print(f'{len(scan)} records scanned')

In [ ]:
# Largest gap in the lambda_1 distribution (all records pooled, no group labels).
lam1 = np.sort(scan['lambda1'].values)
gaps = np.diff(lam1)
gaps[:2] = gaps[-2:] = 0          # require at least 3 records on each side
i = int(np.argmax(gaps))
lower, upper = lam1[i], lam1[i + 1]
print(f'Largest gap in lambda_1: {lower:.3f} to {upper:.3f} (midpoint {(lower + upper) / 2:.3f})')
print(f'Threshold used: {THETA_ARTIFACT_LAMBDA_MIN} | records above: {(lam1 >= THETA_ARTIFACT_LAMBDA_MIN).sum()} of {len(lam1)}')
assert lower < THETA_ARTIFACT_LAMBDA_MIN < upper, 'The threshold is outside the gap; review THETA_ARTIFACT_LAMBDA_MIN.'

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(scan['lambda1'], bins=40, color='0.4')
ax.axvspan(lower, upper, color='0.85', zorder=0, label='gap')
ax.axvline(THETA_ARTIFACT_LAMBDA_MIN, ls='--', color='k', label=f'threshold = {THETA_ARTIFACT_LAMBDA_MIN}')
ax.set(xlabel=r'$\lambda_1$', ylabel='records', title='Narrowband GED, all records')
ax.legend()
plt.tight_layout()
plt.show()

## Step 1 — Preprocess all conditions

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
save_channel_order()

all_logs = []
for condition in CONDITIONS:
    data, logs = process_condition(condition)
    all_logs.extend(logs)
    save_condition(data, condition)

## Step 2 — Log and quality report

In [ ]:
log_path = os.path.join(OUTPUT_DIR, 'processing_log.json')
with open(log_path, 'w', encoding='utf-8') as fh:
    json.dump(all_logs, fh, indent=2)
print(f'Log saved to {log_path}')

quality_report(all_logs)